In [23]:
# --------------------------------------------------
# 1. Imports (add these to the existing ones)
# --------------------------------------------------
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.preprocessing import MultiLabelBinarizer
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import itertools
from torchvision.transforms import Resize
from ast import literal_eval
from sklearn.metrics import matthews_corrcoef
from torchvision import models
from transformers import AutoTokenizer, AutoModelForMaskedLM
import math
import json
from collections import defaultdict
import seaborn as sns
from matplotlib.colors import LogNorm
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
# --------------------------------------------------

In [24]:


class ESM2_Encoder(nn.Module):
    def __init__(self, model_name, trainable=True, unfreeze_last_n=0):
        super().__init__()
        self.esm_mlm = AutoModelForMaskedLM.from_pretrained(model_name)
        self.hidden_size = self.esm_mlm.config.hidden_size
        if not trainable:
            for param in self.esm_mlm.parameters():
                param.requires_grad = False
            if unfreeze_last_n > 0:
                for layer in self.esm_mlm.esm.encoder.layer[-unfreeze_last_n:]:
                    for param in layer.parameters():
                        param.requires_grad = True

    def forward(self, input_ids, attention_mask):
        return self.esm_mlm.esm(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

class SEBlock(nn.Module):
    """
    Mask-aware Squeeze-and-Excitation block.

    Input:
        x        : [B, C, L]
        seq_mask : [B, L], 1 for valid tokens, 0 for PAD

    The channel descriptor is computed using only valid sequence positions.
    """

    def __init__(self, channels, reduction=4):
        super().__init__()

        hidden = max(channels // reduction, 1)

        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x, seq_mask):
        # [B, L] -> [B, 1, L]
        mask = seq_mask.unsqueeze(1).to(dtype=x.dtype)

        # Remove PAD contributions
        x_masked = x * mask

        # Number of valid positions per sequence
        lengths = seq_mask.sum(
            dim=1,
            keepdim=True
        ).clamp_min(1).to(dtype=x.dtype)

        # Masked global average pooling over sequence dimension
        # [B, C, L] -> [B, C]
        channel_descriptor = (
            x_masked.sum(dim=-1) / lengths
        )

        # Channel-wise gates
        gates = self.fc(channel_descriptor).unsqueeze(-1)

        # Apply channel recalibration
        return x * gates

class ConvBlock(nn.Module):
    """
    Mask-aware two-layer 1D convolution block.

    Uses LayerNorm instead of BatchNorm so that padded positions
    do not participate in batch/sequence normalization statistics.
    """

    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.norm1 = nn.LayerNorm(out_channels)

        self.conv2 = nn.Conv1d(
            out_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            bias=False
        )

        self.norm2 = nn.LayerNorm(out_channels)

    @staticmethod
    def apply_layernorm(x, norm):
        """
        Conv output: [B, C, L]
        LayerNorm expects normalized dimension at the end.
        """
        x = x.transpose(1, 2)      # [B, L, C]
        x = norm(x)
        x = x.transpose(1, 2)      # [B, C, L]
        return x

    def forward(self, x, mask):
        """
        x    : [B, C, L]
        mask : [B, 1, L]
        """
        # First convolution
        y = self.conv1(x)
        y = self.apply_layernorm(
            y,
            self.norm1
        )

        y = F.gelu(y)

        # Explicitly remove PAD activations
        y = y * mask

        # Second convolution
        y = self.conv2(y)

        y = self.apply_layernorm(
            y,
            self.norm2
        )

        y = F.gelu(y)

        # Explicitly remove PAD activations again
        y = y * mask

        return y

class EnhancedCNN1D(nn.Module):
    """
    Mask-aware multi-scale CNN for peptide/protein sequences.

    Branches:
        kernel 3 -> effective receptive field 5
        kernel 5 -> effective receptive field 9
        kernel 7 -> effective receptive field 13

    Each branch contains two convolutional layers.

    Output:
        [B, 2 * 3 * conv_dim]

    For conv_dim=128:
        output = [B, 768]
    """

    def __init__(
        self,
        vocab_size=33,
        embed_dim=128,
        conv_dim=128
    ):
        super().__init__()


        # Token embedding
        self.embed = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=1 # ESM TOKENIZER
        )

        # Multi-scale CNN branches
        self.branches = nn.ModuleList([
            ConvBlock(
                in_channels=embed_dim,
                out_channels=conv_dim,
                kernel_size=k
            )
            for k in [3, 5, 7]
        ])

        # Residual projection
        self.res_proj = nn.Conv1d(
            embed_dim,
            conv_dim,
            kernel_size=1,
            bias=False
        )

        # Squeeze-and-Excitation
        self.se = SEBlock(
            channels=conv_dim * 3,
            reduction=4
        )

        self.dropout = nn.Dropout(0.2)

        # Three branches × conv_dim channels
        # Max pooling + mean pooling
        self.hidden_size = conv_dim * 3 * 2

    def forward(self, x, seq_mask):
        """
        Args:
            x:
                [B, L] token IDs

            seq_mask:
                [B, L]
                1 = valid token
                0 = PAD
        Returns:
            CNN feature vector:
                [B, hidden_size]

        For conv_dim=128:
            [B, 768]
        """


        # Validate mask


        if seq_mask is None:
            raise ValueError(
                "seq_mask must be provided to EnhancedCNN1D. "
                "CNN padding masking must never be bypassed."
            )
        # Embedding
        # [B, L] -> [B, L, embed_dim]
        x = self.embed(x)

        # [B, L, embed_dim] -> [B, embed_dim, L]
        x = x.transpose(1, 2)

        # [B, L] -> [B, 1, L]
        mask = seq_mask.unsqueeze(1).to(dtype=x.dtype)

        # Explicitly zero PAD embeddings
        x = x * mask

        # Residual pathway
        res = self.res_proj(x)

        # Remove PAD activations
        res = res * mask

        # Multi-scale branches
        outs = []

        for branch in self.branches:
            y = branch(x, mask)
            # Residual connection
            y = y + res
            # Guarantee PAD = 0 after residual addition
            y = y * mask
            outs.append(y)

        # Concatenate branches
        # [B, 128*3, L]
        combined = torch.cat(
            outs,
            dim=1
        )

        # Guarantee no PAD signal
        combined = combined * mask

        # Mask-aware SE
        combined = self.se(
            combined,
            seq_mask
        )

        # SE can theoretically produce nonzero values
        # at PAD positions, so mask once more.
        combined = combined * mask

        # MASKED GLOBAL MAX POOLING
        # PAD cannot become the maximum.
        combined_for_max = combined.masked_fill(
            seq_mask.unsqueeze(1) == 0,
            torch.finfo(combined.dtype).min
        )

        max_pooled = combined_for_max.max(
            dim=-1
        ).values


        # MASKED GLOBAL MEAN POOLING
        combined_for_mean = combined * mask

        # Actual number of valid tokens
        lengths = seq_mask.sum(
            dim=1,
            keepdim=True
        ).clamp_min(1).to(
            dtype=combined.dtype
        )

        mean_pooled = (
            combined_for_mean.sum(dim=-1)
            / lengths
        )

        # FINAL REPRESENTATION
        pooled = torch.cat(
            [
                max_pooled,
                mean_pooled
            ],
            dim=1
        )

        return self.dropout(pooled)


class MultiHeadAttentionPool(nn.Module):
    def __init__(self, dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.attn = nn.Sequential(
            nn.Linear(dim, 128), 
            nn.Tanh(), 
            nn.Linear(128, num_heads)
        )
        self._last_weights = None

    def forward(self, x, mask):
        scores = self.attn(x)
        scores = scores.masked_fill(mask.unsqueeze(-1) == 0, -1e4)
        weights = torch.softmax(scores, dim=1)
        self._last_weights = weights
        pooled = (x.unsqueeze(2) * weights.unsqueeze(-1)).sum(dim=1)
        return pooled.view(x.size(0), -1)


    def orthogonality_loss(self):
        """Penalize overlap between head attention distributions."""
        if self._last_weights is None:
            return 0.0
            
        # weights: [B, L, num_heads] -> transpose to [B, num_heads, L]
        w = self._last_weights.transpose(1, 2)  
        
        # Gram matrix of head attention distributions: shape [B, num_heads, num_heads]
        gram = torch.bmm(w, w.transpose(1, 2))  
        
        # Create a boolean mask for the off-diagonal elements (~torch.eye inverts the identity matrix)
        mask = ~torch.eye(self.num_heads, dtype=torch.bool, device=gram.device)
        
        # Penalize only the off-diagonal overlap to encourage heads to focus on different tokens.
        # We want these dot products to be pushed towards 0.
        loss = (gram[:, mask] ** 2).mean()
        
        return loss
    
class GatedFusion(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(input_dim, input_dim), nn.Sigmoid())
    def forward(self, x):
        return x * self.gate(x)

class PeptideNetwork(nn.Module):
    def __init__(self, num_classes=21, mask_token_id=32):
        super().__init__()
        self.mask_token_id = mask_token_id
        self.num_classes = num_classes

        # BOTH encoders are now the 8M parameter t6 model
        self.esm_t6_a = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=True)        
        self.esm_t6_b = ESM2_Encoder("facebook/esm2_t6_8M_UR50D", trainable=False, unfreeze_last_n=2)                                  
        self.cnn = EnhancedCNN1D()          # Bx768

        # REDUCED cross_dim to 128 to save parameters
        cross_dim = 128
        self.proj_t6_a = nn.Linear(320, cross_dim)
        self.proj_t6_b = nn.Linear(320, cross_dim) # Updated to 320 for t6

        self.cross_t6_a = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_a = nn.LayerNorm(cross_dim)
        
        self.cross_t6_b = nn.MultiheadAttention(cross_dim, num_heads=4, batch_first=True, dropout=0.1)
        self.ln_ca_t6_b = nn.LayerNorm(cross_dim)
        
        self.pool_t6_a = MultiHeadAttentionPool(cross_dim, num_heads=4)
        self.pool_t6_b = MultiHeadAttentionPool(cross_dim, num_heads=4)

        # Bottleneck reduction layer before fusion
        # Concat size: 128*4*3 (pools) + 768 (CNN) = 1536 + 768 = 2304
        concat_size = cross_dim * 4 * 2 + self.cnn.hidden_size
        
        self.dim_reduce = nn.Sequential(
            nn.Linear(concat_size, 512),
            nn.GELU()
        )
        
        # Fusion now operates efficiently on 512 dimensions
        self.fusion = GatedFusion(512)
        self.ln = nn.LayerNorm(512)

        # binary head
        binary_features_dim = 64
        self.binary_features = nn.Sequential(
            nn.Linear(512, binary_features_dim),
            nn.GELU(),
            nn.Dropout(0.2)
        )
        self.binary_classifier = nn.Linear(64, 1) # Final logit

        # Task Query Decoder
        self.task_dim = 128
        self.n_memory_tokens = 16
        self.memory_proj = nn.Sequential(
            nn.Linear(512 + binary_features_dim, self.task_dim * self.n_memory_tokens),
            nn.GELU(),
        )
        self.task_queries = nn.Embedding(num_classes, self.task_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=self.task_dim, nhead=4, dim_feedforward=256,
            batch_first=True, dropout=0.1
        )
        self.task_decoder = nn.TransformerDecoder(decoder_layer, num_layers=2)
        self.task_classifiers = nn.ModuleList([
            nn.Linear(self.task_dim, 1) for _ in range(num_classes)
        ])

    def _mask_tokens(self, input_ids, attention_mask, mask_prob=0.15):
        masked_ids = input_ids.clone()
        prob_matrix = torch.full_like(input_ids, mask_prob, dtype=torch.float)
        prob_matrix[attention_mask == 0] = 0
        prob_matrix[:, 0] = 0
        seq_lens = attention_mask.sum(dim=1)
        for i in range(len(seq_lens)):
            if seq_lens[i] > 1:
                prob_matrix[i, seq_lens[i] - 1] = 0
        mask = torch.bernoulli(prob_matrix).bool()
        masked_ids[mask] = self.mask_token_id
        return masked_ids

    def _extract_features(self, seq_input, seq_mask):
        esm6_a_seq = self.esm_t6_a(seq_input, seq_mask)       
        esm6_b_seq = self.esm_t6_b(seq_input, seq_mask)     
        cnn_feat = self.cnn(seq_input, seq_mask)                            

        t6_a = self.proj_t6_a(esm6_a_seq)       
        t6_b = self.proj_t6_b(esm6_b_seq)    


        kv_pad = seq_mask == 0  

        ca_t6_a, _ = self.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_t6_a = self.ln_ca_t6_a(t6_a + ca_t6_a)

        ca_t6_b, _ = self.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
        ca_t6_b = self.ln_ca_t6_b(t6_b + ca_t6_b)

        pooled_t6_a = self.pool_t6_a(ca_t6_a, seq_mask)
        pooled_t6_b = self.pool_t6_b(ca_t6_b, seq_mask)

        # Concat -> Reduce -> Fuse
        combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)  
        reduced = self.dim_reduce(combined)
        fusion = self.ln(reduced + self.fusion(reduced))

        binary_features = self.binary_features(fusion)
        
        return binary_features, torch.cat([fusion, binary_features], dim=1)
    
    def _binary_classify(self, binary_features):
        
        binary_logits = self.binary_classifier(binary_features)

        return binary_logits

    def _classify(self, final_fusion):
        B = final_fusion.size(0)
        memory = self.memory_proj(final_fusion).view(B, self.n_memory_tokens, self.task_dim)
        tgt = self.task_queries.weight.unsqueeze(0).expand(B, -1, -1)
        decoded = self.task_decoder(tgt, memory)
        logits = torch.cat([self.task_classifiers[i](decoded[:, i, :])
                           for i in range(self.num_classes)], dim=1)
        return logits

    def forward(self, seq_input, seq_mask, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        
        binary_features, combined_features = self._extract_features(seq_input, seq_mask)
        

        return self._binary_classify(binary_features), self._classify(combined_features)


    def ortho_loss(self):
        return (self.pool_t6_a.orthogonality_loss() +
                self.pool_t6_b.orthogonality_loss() 
                ) / 2
    
    def get_features(self, seq_input, seq_mask, mask_tokens=False):
        if mask_tokens and self.training:
            seq_input = self._mask_tokens(seq_input, seq_mask)
        return self._extract_features(seq_input, seq_mask)

    def multi_classify(self, combined):
        return self._classify(combined)
    
    def binary_classify(self, binary_features):
        return self._binary_classify(binary_features)



In [25]:
def enable_dropout(model):
    """
    Specifically tailored for PeptideNetwork to activate all sources of dropout
    during Test-Time Augmentation (TTA), including functional dropouts hidden 
    inside complex PyTorch modules.
    """
    for module in model.modules():
        class_name = module.__class__.__name__
        
        # 1. Standard explicit dropout layers (Dropout, Dropout1d, Dropout2d)
        if class_name.startswith('Dropout'):
            module.train()
            
        # 2. Cross-Attention functional dropout
        elif class_name == 'MultiheadAttention':
            module.train()
            
        # 3. Task Query Decoder functional dropout
        elif class_name in ['TransformerDecoder', 'TransformerDecoderLayer']:
            module.train()
            
        # 4. GRU functional dropout (applied between internal layers)
        elif class_name == 'GRU':
            module.train()

In [26]:
esm_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")

In [27]:
DEVICE = 'cuda:0'

In [28]:

# Load Reward Model globally on REWARD_DEVICE
print(f"Loading Model on {DEVICE}...")
model = PeptideNetwork(num_classes=13, mask_token_id=32)
# Ensure the model exists at this location
model_path = "../../phase_2_model.pt"
if os.path.exists(model_path):
    print(f"Found {model_path}! Loading weights.")
    model.load_state_dict(torch.load(model_path, map_location="cpu"))
else:
    raise Exception(f"Warning: {model_path} not found. model is untrained.")
model = model.to(DEVICE)
model.eval()

# Endpoints matching classification_training.py (excluding non-functional from specific list)
endpoints_functional = [
    'anti-bacterial', 'anti-cancer', 'anti-fungal', 'anti-parasitic', 'anti-viral', 
    'cell-cell-communication', 'drug-delivery', 'immunological', 'inhibitor', 
    'metabolic', 'other-functional', 'signal-peptide', 'toxic'
]


THRESHOLD_PATH = '../../phase_2_model.json'
# reading threasholds 
with open(THRESHOLD_PATH) as f:
    data = json.load(f)
    best_i = 0
    for i, e in enumerate(data):
        if e['val_avg_func_mcc'] > data[best_i]['val_avg_func_mcc']:
            best_i = i
    
    print(f'reading thresholds from {best_i}th epoch, as it gave the best val mcc')
    threshold_binary = data[best_i]['thresholds']['binary'][0]
    threshold_functional = data[best_i]['thresholds']['functional']
    threshold_functional = {endpoints_functional[i]: threshold_functional[i] for i in range(len(endpoints_functional))}

thresholds = {**threshold_functional, 'non-functional': threshold_binary}


def batched_sequence_classifier(sequences: list[str], model, batch_size=256, tta_passes=1) -> list[dict]:
    """
    Batched inference for external multi-label sequence classifier.
    Returns list of dicts with calculated probabilities for 'target_avg', 'toxic'
    """
    if not sequences: 
        return []
    
    test_tta_bin_preds = [[] for _ in range(tta_passes)]
    test_tta_spec_preds = [[] for _ in range(tta_passes)]

    with torch.no_grad():
        model.eval()
        for i in range(0, len(sequences), batch_size):
            batch = sequences[i: i+batch_size]
            
            # FIX: Tokenize the 'batch', not the entire 'sequences' list!
            encodings = esm_tokenizer(batch, add_special_tokens=True, max_length=100,
                                        padding='max_length', truncation=True, return_attention_mask=True, return_tensors='pt')

            input_ids = encodings['input_ids'].to(DEVICE)
            attention_mask = encodings['attention_mask'].to(DEVICE)


            pred_bin, pred_spec = model(input_ids, attention_mask)

            # First pass predictions
            test_tta_bin_preds[0].append(torch.sigmoid(pred_bin).detach().cpu().numpy())
            test_tta_spec_preds[0].append(torch.sigmoid(pred_spec).detach().cpu().numpy())

            # Additional TTA passes
            if tta_passes > 1:
                enable_dropout(model)
                for t in range(1, tta_passes):
                    pred_bin_t, pred_spec_t = model(input_ids, attention_mask)
                    test_tta_bin_preds[t].append(torch.sigmoid(pred_bin_t).detach().cpu().numpy())
                    test_tta_spec_preds[t].append(torch.sigmoid(pred_spec_t).detach().cpu().numpy())
                model.eval()

    y_pred_bin_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_bin_preds], axis=0)

    raw_pred_spec_test = np.mean([np.concatenate(pred_list) for pred_list in test_tta_spec_preds], axis=0)

    # SOFT GATING for Test
    y_pred_spec_test = raw_pred_spec_test * (1 - y_pred_bin_test)


    results = []
    for bp, sp in zip(y_pred_bin_test, y_pred_spec_test):

        
        gated_probs_dict = {ep: p.item() for ep, p in zip(endpoints_functional, sp)}
        gated_probs_dict['non-functional'] = bp[0].item()

        results.append({
            'gated_probs': gated_probs_dict,

        })

    return results

Loading Model on cuda:0...


Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

Found ../../phase_2_model.pt! Loading weights.
reading thresholds from 21th epoch, as it gave the best val mcc


In [29]:
def residue_saliency_dual(model, seq_input, seq_mask, target_class=0):
    model.eval()

    # --- 1. Get BOTH ESM hidden states as differentiable leaf tensors ---
    esm_a_hidden = model.esm_t6_a(seq_input, seq_mask)
    esm_a_hidden = esm_a_hidden.clone().detach().requires_grad_(True)

    esm_b_hidden = model.esm_t6_b(seq_input, seq_mask)
    esm_b_hidden = esm_b_hidden.clone().detach().requires_grad_(True) # NEW

    # --- 2. CNN (no grad needed, but seq_mask is MANDATORY for this CNN) ---
    with torch.no_grad():
        cnn_feat = model.cnn(seq_input, seq_mask)

    # --- 3. Project to cross-attention dim ---
    t6_a = model.proj_t6_a(esm_a_hidden)
    t6_b = model.proj_t6_b(esm_b_hidden)

    # --- 4. Cross-attention ---
    kv_pad = seq_mask == 0
    ca_t6_a, _ = model.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
    ca_t6_a = model.ln_ca_t6_a(t6_a + ca_t6_a)

    ca_t6_b, _ = model.cross_t6_b(t6_b, t6_a, t6_a, key_padding_mask=kv_pad)
    ca_t6_b = model.ln_ca_t6_b(t6_b + ca_t6_b)

    # --- 5. Multi-head pooling ---
    pooled_t6_a = model.pool_t6_a(ca_t6_a, seq_mask)
    pooled_t6_b = model.pool_t6_b(ca_t6_b, seq_mask)

    # --- 6. Fusion ---
    combined = torch.cat([pooled_t6_a, pooled_t6_b, cnn_feat], dim=1)
    reduced = model.dim_reduce(combined)
    fusion = model.ln(reduced + model.fusion(reduced))

    # --- 7. Classify ---
    binary_features = model.binary_features(fusion)
    combined_features = torch.cat([fusion, binary_features], dim=1)
    specific_logits = model._classify(combined_features)

    # --- 8. Backward from target class score ---
    score = specific_logits[0, target_class]
    model.zero_grad()
    score.backward()

    # --- 9. Aggregate gradients for BOTH ---
    if esm_a_hidden.grad is None or esm_b_hidden.grad is None:
        raise RuntimeError(
            "Saliency gradients are None — backward() did not reach "
            "the ESM hidden states."
        )

    saliency_a = esm_a_hidden.grad.abs().sum(dim=-1).squeeze().cpu().numpy()
    saliency_a = (saliency_a - saliency_a.min()) / (saliency_a.max() - saliency_a.min() + 1e-8)

    saliency_b = esm_b_hidden.grad.abs().sum(dim=-1).squeeze().cpu().numpy()
    saliency_b = (saliency_b - saliency_b.min()) / (saliency_b.max() - saliency_b.min() + 1e-8)

    return saliency_a, saliency_b

In [30]:
def get_pooling_attention(model, seq_input, seq_mask):
    """
    Returns: [num_heads, seq_len] — attention weight per head per position
    """
    model.eval()
    with torch.no_grad():
        # Forward through encoders and cross-attention
        esm_a = model.esm_t6_a(seq_input, seq_mask)
        esm_b = model.esm_t6_b(seq_input, seq_mask)
        t6_a = model.proj_t6_a(esm_a)
        t6_b = model.proj_t6_b(esm_b)
        
        kv_pad = seq_mask == 0
        ca_a, _ = model.cross_t6_a(t6_a, t6_b, t6_b, key_padding_mask=kv_pad)
        ca_a = model.ln_ca_t6_a(t6_a + ca_a)
        
        # Trigger pool forward to populate _last_weights
        _ = model.pool_t6_a(ca_a, seq_mask)
        attn = model.pool_t6_a._last_weights  # [B, T, H]
        
    return attn[0].transpose(0, 1).cpu().numpy()  # [H, T]

# USAGE
# attn = get_pooling_attention(model, seq_input, seq_mask)
# plt.imshow(attn, aspect='auto', cmap='hot')
# plt.yticks([0,1,2,3], ['Head 1','Head 2','Head 3','Head 4'])

In [31]:
def get_cross_attention_map(model, seq_input, seq_mask):
    """
    Returns: [num_heads, seq_len, seq_len] 
    Shows where trainable ESM looks in frozen ESM
    """
    model.eval()
    with torch.no_grad():
        esm_a = model.esm_t6_a(seq_input, seq_mask)
        esm_b = model.esm_t6_b(seq_input, seq_mask)
        t6_a = model.proj_t6_a(esm_a)
        t6_b = model.proj_t6_b(esm_b)
        
        kv_pad = seq_mask == 0
        
        # need_weights=True returns attention matrix
        _, attn_weights = model.cross_t6_a(
            t6_a, t6_b, t6_b,
            key_padding_mask=kv_pad,
            need_weights=True,
            average_attn_weights=False
        )
        # attn_weights: [B, num_heads, T, T]
    return attn_weights[0].cpu().numpy()  # [H, T, T]

In [32]:
"""
CNN Motif Extraction & Visualization
=====================================
Replaces the naive `sum(dim=0)` activation trace with:
1. Per-filter top-k patch extraction  -> sequence logos (matches manuscript)
2. Class-weighted (Grad-CAM) per-position importance  -> biologically meaningful traces

Compatible with EnhancedCNN1D / ConvBlock where branches are called as branch(x, mask).
"""

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from collections import Counter
import math
from matplotlib.patches import Patch


# ---------------------------------------------------------------------------
# 1. EXTRACT TOP-K PATCHES PER FILTER (Manuscript-compliant)
# ---------------------------------------------------------------------------

def extract_topk_patches(branch_out, sequence, kernel_size, aa_start, aa_end,
                         top_k=10, min_act_percentile=50):
    """
    Extract top-k activating sequence patches for every filter in a branch.

    Parameters
    ----------
    branch_out : Tensor [1, C, L]
        Raw conv output BEFORE residual addition.
    sequence : str
        Amino-acid sequence (no special tokens).
    kernel_size : int
        Receptive field of this branch (3, 5, or 7).
    aa_start, aa_end : int
        Slicing indices that bound the amino-acid region in branch_out.
    top_k : int
        Number of peak positions to retain per filter.
    min_act_percentile : float
        Ignore filters whose max activation is below this percentile (removes dead filters).

    Returns
    -------
    list of dicts, one per active filter:
        {
            'filter_id': int,
            'kernel_size': int,
            'max_activation': float,
            'patches': list of str   # length == top_k (or fewer near edges)
        }
    """
    C = branch_out.shape[1]
    activations = branch_out[0, :, aa_start:aa_end]  # [C, aa_len]
    aa_len = aa_end - aa_start
    L_seq = len(sequence)

    # Global threshold to skip dead / noisy filters
    global_max = activations.max().item()
    act_threshold = np.percentile(activations.cpu().numpy(), min_act_percentile)

    filter_data = []
    half_k = kernel_size // 2

    for c in range(C):
        filt_act = activations[c]  # [aa_len]
        max_val = filt_act.max().item()

        if max_val < act_threshold or max_val <= 0:
            continue

        # Top-k positions (amino-acid coordinates)
        topk_vals, topk_idx = torch.topk(filt_act, min(top_k, aa_len))
        topk_idx = topk_idx.cpu().numpy()

        patches = []
        for pos in topk_idx:
            # Map conv position -> sequence index
            # Conv output at position i corresponds to kernel centered at i in the aa-region
            seq_center = pos
            if seq_center < 0 or seq_center >= L_seq:
                continue
            start = max(0, seq_center - half_k)
            end = min(L_seq, seq_center + half_k + 1)
            patch = sequence[start:end]
            # Pad if near termini so all patches have same length
            left_pad = half_k - (seq_center - start)
            right_pad = half_k - (end - 1 - seq_center)
            patch = '-' * left_pad + patch + '-' * right_pad
            patches.append(patch)

        if len(patches) >= 3:  # Need enough patches to build a meaningful logo
            filter_data.append({
                'filter_id': c,
                'kernel_size': kernel_size,
                'max_activation': max_val,
                'patches': patches
            })

    # Sort by max activation descending
    filter_data.sort(key=lambda x: x['max_activation'], reverse=True)
    return filter_data


# ---------------------------------------------------------------------------
# 2. SEQUENCE LOGO RENDERER (pure matplotlib — no extra deps)
# ---------------------------------------------------------------------------

AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY-")
AA_COLORS = {
    'A': '#c8c8c8', 'C': '#e6e600', 'D': '#e60a0a', 'E': '#e60a0a',
    'F': '#3232aa', 'G': '#c8c8c8', 'H': '#8282d2', 'I': '#0f820f',
    'K': '#145aff', 'L': '#0f820f', 'M': '#0f820f', 'N': '#00dcdc',
    'P': '#c8c8c8', 'Q': '#00dcdc', 'R': '#145aff', 'S': '#fa9600',
    'T': '#fa9600', 'V': '#0f820f', 'W': '#3232aa', 'Y': '#3232aa',
    '-': '#ffffff'
}

# Grouped legend entries: (label, color_hex, amino_acids)
LEGEND_GROUPS = [
    ("Small / nonpolar",    "#c8c8c8", "A, G, P"),
    ("Acidic (-)",          "#e60a0a", "D, E"),
    ("Aromatic",            "#3232aa", "F, W, Y"),
    ("Basic (+)",           "#145aff", "H, K, R"),
    ("Aliphatic",           "#0f820f", "I, L, M, V"),
    ("Polar / hydroxyl",    "#fa9600", "S, T"),
    ("Amide",               "#00dcdc", "N, Q"),
]

def build_pwm(patches):
    """Build position weight matrix from aligned patches."""
    if not patches:
        return None
    width = len(patches[0])
    pwm = np.zeros((width, len(AA_ORDER)))
    for p in patches:
        for i, aa in enumerate(p):
            if aa in AA_ORDER:
                pwm[i, AA_ORDER.index(aa)] += 1
    # Normalize to frequencies
    pwm = pwm / pwm.sum(axis=1, keepdims=True)
    # Small pseudocount
    pwm = (pwm + 1e-4) / (1 + len(AA_ORDER) * 1e-4)
    return pwm


def draw_sequence_logo(patches, ax=None, title="", ylabel="Bits"):
    """
    Draw a sequence logo from aligned patches.
    Height = frequency (can be swapped to information content if desired).
    """
    pwm = build_pwm(patches)
    if pwm is None:
        return None

    if ax is None:
        fig, ax = plt.subplots(figsize=(len(patches[0]) * 0.6, 2.5))
    else:
        fig = ax.figure

    width = pwm.shape[0]
    x_positions = np.arange(width)

    for i in range(width):
        col = pwm[i]
        # Sort amino acids by frequency descending
        order = np.argsort(-col)
        y_offset = 0.0
        for idx in order:
            freq = col[idx]
            if freq < 0.02:
                continue
            aa = AA_ORDER[idx]
            color = AA_COLORS.get(aa, '#333333')
            ax.bar(x_positions[i], freq, bottom=y_offset, color=color,
                   edgecolor='black', linewidth=0.3, width=0.9)
            # Add letter if bar is tall enough
            if freq > 0.10:
                ax.text(x_positions[i], y_offset + freq / 2, aa,
                        ha='center', va='center', fontsize=9,
                        color='white' if aa in 'DEHKR' else 'black',
                        fontweight='bold')
            y_offset += freq

    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(0, 1.1)
    ax.set_xticks(x_positions)
    ax.set_xticklabels([str(i + 1) for i in x_positions], fontsize=8)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    fig.tight_layout()
    return fig


def add_chemical_legend(fig, ncols=4):
    """
    Add a horizontal legend bar at the bottom of the figure
    showing chemical class -> color mapping.
    """
    legend_elements = [
        Patch(facecolor=color, edgecolor='black', linewidth=0.5,
              label=f"{label}  ({aas})")
        for label, color, aas in LEGEND_GROUPS
    ]
    fig.legend(
        handles=legend_elements,
        loc='lower center',
        ncol=ncols,
        fontsize=9,
        frameon=True,
        fancybox=False,
        edgecolor='black',
        bbox_to_anchor=(0.5, -0.02)
    )


# ---------------------------------------------------------------------------
# 3. MAIN PLOT: Motif Grid (what the manuscript actually describes)
# ---------------------------------------------------------------------------


def plot_cnn_motif_logos(model, seq_input, sequence, seq_len,
                         top_k=15, n_filters_per_branch=4):
    L = len(sequence)
    if L == 0 or seq_len < 3:
        return None

    model.eval()
    kernels = [3, 5, 7]

    with torch.no_grad():
        x = model.cnn.embed(seq_input).transpose(1, 2)
        mask = torch.zeros_like(seq_input, dtype=torch.float)
        mask[:, :seq_len] = 1.0
        mask_3d = mask.unsqueeze(1)
        x = x * mask_3d

        aa_start = 1
        aa_end = seq_len - 1

        all_branch_data = []
        for i, branch in enumerate(model.cnn.branches):
            branch_out = branch(x, mask_3d)
            # ---- extract top-k patches per filter ----
            filter_data = []
            C = branch_out.shape[1]
            activations = branch_out[0, :, aa_start:aa_end]
            aa_len = aa_end - aa_start
            half_k = kernels[i] // 2
            global_max = activations.max().item()
            act_threshold = np.percentile(activations.cpu().numpy(), 50)

            for c in range(C):
                filt_act = activations[c]
                max_val = filt_act.max().item()
                if max_val < act_threshold or max_val <= 0:
                    continue
                topk_vals, topk_idx = torch.topk(filt_act, min(top_k, aa_len))
                topk_idx = topk_idx.cpu().numpy()
                patches = []
                for pos in topk_idx:
                    seq_center = pos
                    if seq_center < 0 or seq_center >= L:
                        continue
                    start = max(0, seq_center - half_k)
                    end = min(L, seq_center + half_k + 1)
                    patch = sequence[start:end]
                    left_pad = half_k - (seq_center - start)
                    right_pad = half_k - (end - 1 - seq_center)
                    patch = '-' * left_pad + patch + '-' * right_pad
                    patches.append(patch)
                if len(patches) >= 3:
                    filter_data.append({
                        'filter_id': c,
                        'kernel_size': kernels[i],
                        'max_activation': max_val,
                        'patches': patches
                    })
            filter_data.sort(key=lambda x: x['max_activation'], reverse=True)
            all_branch_data.append(filter_data)

    # ---- Plotting with legend ----
    n_branches = len(model.cnn.branches)
    fig, axes = plt.subplots(n_branches, n_filters_per_branch,
                             figsize=(n_filters_per_branch * 2.4, n_branches * 2.8 + 1.2))
    if n_branches == 1:
        axes = axes.reshape(1, -1)

    for b_idx, filter_data in enumerate(all_branch_data):
        for f_idx in range(n_filters_per_branch):
            ax = axes[b_idx, f_idx]
            if f_idx < len(filter_data):
                fd = filter_data[f_idx]
                title = f"k={fd['kernel_size']}  F{fd['filter_id']}  (max={fd['max_activation']:.2f})"
                draw_sequence_logo(fd['patches'], ax=ax, title=title)
            else:
                ax.axis('off')

    fig.suptitle(f"CNN Motif Extraction — {sequence[:25]}...", fontsize=13, fontweight='bold')

    # ---- ADD LEGEND ----
    add_chemical_legend(fig, ncols=4)

    # Extra bottom space for legend
    fig.tight_layout(rect=[0, 0.08, 1, 0.95])
    return fig_to_pil(fig)


# ---------------------------------------------------------------------------
# 4. IMPROVED PER-POSITION PLOT: Abs-sum + Class-weighted (Grad-CAM style)
# ---------------------------------------------------------------------------

def plot_cnn_activations_improved(model, seq_input, sequence, seq_len,
                                  target_class_idx=None,
                                  use_abs=True,
                                  use_class_weight=False):
    """
    Improved per-position CNN activation plot.

    Options:
    - use_abs=True        : sum of |activations|  (no cancellation)
    - use_class_weight=True: weight each channel by its gradient importance
                             for target_class_idx (requires target_class_idx).
    """
    L = len(sequence)
    if L == 0 or seq_len < 3:
        return None

    aa_start = 1
    aa_end = seq_len - 1
    display_len = min(L, aa_end - aa_start)

    model.eval()
    kernels = [3, 5, 7]
    colors = ['#e74c3c', '#2ecc71', '#3498db']

    x = model.cnn.embed(seq_input).transpose(1, 2)
    mask = torch.zeros_like(seq_input, dtype=torch.float)
    mask[:, :seq_len] = 1.0
    mask_3d = mask.unsqueeze(1)
    x = x * mask_3d

    fig, ax = plt.subplots(figsize=(max(10, display_len * 0.35), 4))

    for i, branch in enumerate(model.cnn.branches):
        branch_out = branch(x, mask_3d)  # [1, C, L]

        if use_class_weight and target_class_idx is not None:
            # ---- Grad-CAM style weighting ----
            branch_out_req = branch_out.requires_grad_(True)
            # Forward through rest of model (simplified; assumes you can hook)
            # NOTE: This is a template — actual implementation depends on your
            # fusion + decoder architecture. Below is the conceptual flow.
            raise NotImplementedError(
                "Class-weighted mode requires a forward hook through your "
                "fusion/gatekeeper/decoder. Use plot_cnn_gradcam() instead."
            )
        else:
            # ---- Abs-sum (no cancellation) ----
            if use_abs:
                act = branch_out[0, :, aa_start:aa_end].abs().sum(dim=0)
            else:
                act = branch_out[0, :, aa_start:aa_end].sum(dim=0)
            aa_act = act.detach().cpu().numpy()[:display_len]

        positions = np.arange(display_len)
        ax.plot(positions, aa_act, color=colors[i], linewidth=1.5,
                marker='o', markersize=3, label=f'Kernel {kernels[i]}', alpha=0.85)

    tick_step = max(1, display_len // 20)
    tick_positions = np.arange(0, display_len, tick_step)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([sequence[i] for i in tick_positions], fontsize=7)
    ax.set_xlabel('Amino-acid position', fontsize=11)
    ylabel = 'CNN activation (sum of |channels|)' if use_abs else 'CNN activation (sum over channels)'
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title('CNN Multi-Scale Motif Activations (Amino Acids Only)', fontsize=13, fontweight='bold')
    ax.axhline(0, linewidth=0.8, alpha=0.4)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    return fig_to_pil(fig)


# ---------------------------------------------------------------------------
# 5. FULL GRAD-CAM (Optional advanced usage)
# ---------------------------------------------------------------------------

def plot_cnn_gradcam(model, seq_input, sequence, seq_len, target_class_idx):
    """
    Class Activation Mapping for CNN branches.
    Weights each filter by gradient of target_class logit w.r.t. that filter.
    """
    L = len(sequence)
    if L == 0 or seq_len < 3:
        return None

    model.eval()
    aa_start = 1
    aa_end = seq_len - 1
    display_len = min(L, aa_end - aa_start)
    kernels = [3, 5, 7]
    colors = ['#e74c3c', '#2ecc71', '#3498db']

    fig, ax = plt.subplots(figsize=(max(10, display_len * 0.35), 4))

    for i, branch in enumerate(model.cnn.branches):
        x = model.cnn.embed(seq_input).transpose(1, 2)
        mask = torch.zeros_like(seq_input, dtype=torch.float)
        mask[:, :seq_len] = 1.0
        mask_3d = mask.unsqueeze(1)
        x = x * mask_3d

        branch_out = branch(x, mask_3d)
        branch_out = branch_out.requires_grad_(True)

        # --- Forward through remaining architecture ---
        # You must adapt this block to your exact fusion/decoder pipeline.
        # Below is a schematic assuming:
        #   branch_out -> fusion -> binary gatekeeper -> task decoder -> logits
        fused = model.fusion(branch_out)  # PLACEHOLDER
        logits = model.decoder(fused)      # PLACEHOLDER
        target_score = logits[0, target_class_idx]

        model.zero_grad()
        target_score.backward(retain_graph=True)

        grad = branch_out.grad[0, :, aa_start:aa_end]  # [C, aa_len]
        act = branch_out[0, :, aa_start:aa_end].detach()  # [C, aa_len]

        # Global average pooling of gradients per filter -> weights
        weights = grad.mean(dim=1, keepdim=True)  # [C, 1]
        cam = (weights * act).sum(dim=0).cpu().numpy()  # [aa_len]
        cam = cam[:display_len]

        # ReLU + normalize
        cam = np.maximum(cam, 0)
        if cam.max() > 0:
            cam = cam / cam.max()

        positions = np.arange(display_len)
        ax.plot(positions, cam, color=colors[i], linewidth=1.5,
                marker='o', markersize=3, label=f'Kernel {kernels[i]}', alpha=0.85)

    tick_step = max(1, display_len // 20)
    tick_positions = np.arange(0, display_len, tick_step)
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([sequence[i] for i in tick_positions], fontsize=7)
    ax.set_xlabel('Amino-acid position', fontsize=11)
    ax.set_ylabel('Class-weighted importance (Grad-CAM)', fontsize=11)
    ax.set_title(f'CNN Class Activation — Class {target_class_idx}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    return fig_to_pil(fig)


# ---------------------------------------------------------------------------
# 6. DROP-IN REPLACEMENT for your old function
# ---------------------------------------------------------------------------

def plot_cnn_activations(model, seq_input, sequence, seq_len, mode="logos"):
    """
    Unified entry point. Set mode:
        "logos"  -> manuscript-compliant sequence logos (RECOMMENDED)
        "abs"    -> improved line plot with |activation| summing
        "gradcam"-> class-weighted importance (requires target_class_idx)
    """
    if mode == "logos":
        return plot_cnn_motif_logos(model, seq_input, sequence, seq_len)
    elif mode == "abs":
        return plot_cnn_activations_improved(model, seq_input, sequence, seq_len,
                                             use_abs=True, use_class_weight=False)
    elif mode == "gradcam":
        # Requires additional target_class_idx argument in practice
        raise ValueError("Use plot_cnn_gradcam() directly for Grad-CAM mode.")
    else:
        raise ValueError(f"Unknown mode: {mode}")

In [32]:
# =====================================================================
# 5. Gradio Explainability Dashboard Server
# =====================================================================

# !pip install gradio -q

%load_ext autoreload
%autoreload 2

import gradio as gr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
import numpy as np
import textwrap
import pandas as pd

# ---------- Figure → PIL Image Helper ----------
def fig_to_pil(fig, dpi=130):
    """Convert a matplotlib Figure to a PIL Image for Gradio."""
    buf = BytesIO()
    fig.savefig(buf, format='png', dpi=dpi, bbox_inches='tight',
                pad_inches=0.25,
                facecolor='white', edgecolor='none')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img


# ---------- Visualization Functions ----------

def plot_predictions(gated_probs):
    """Horizontal bar chart: class probabilities with threshold indicators."""
    classes = list(gated_probs.keys())
    probs  = np.array(list(gated_probs.values()))

    sort_idx = np.argsort(probs)[::-1]
    classes = [classes[i] for i in sort_idx]
    probs   = probs[sort_idx]

    colors = []
    for c, p in zip(classes, probs):
        th = thresholds.get(c, 0.5)
        colors.append('#27ae60' if p >= th else '#c0392b')

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(classes, probs, color=colors, edgecolor='white', linewidth=0.5, height=0.6)
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_width() + 0.015, bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9, fontweight='bold')
    ax.axvline(x=0.5, color='#7f8c8d', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.set_xlim(0, 1.12)
    ax.set_xlabel('Probability', fontsize=11)
    ax.set_title('Predicted Functional Class Probabilities', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    fig.tight_layout()
    return fig_to_pil(fig)




def plot_saliency(
    sequence,
    saliency_a,
    saliency_b,
    seq_len,
    top_k=5,
    palette=("teal", "coral"),
):
    """Plot publication-ready residue-level gradient saliency for dual ESM branches.

    Parameters
    ----------
    sequence : str
        Amino acid sequence string.
    saliency_a : array-like
        Residue gradient saliency from ESM Branch A (Trainable).
    saliency_b : array-like
        Residue gradient saliency from ESM Branch B (Frozen).
    seq_len : int
        Total sequence length including special tokens ([CLS], [EOS], [PAD]).
    top_k : int, default=5
        Number of top salient residues to highlight per branch.
    palette : tuple or str, default=('teal', 'coral')
        Visual theme for Branch A and B ('teal', 'blue', 'purple', etc.).

    Returns
    -------
    PIL.Image or matplotlib.figure.Figure
    """
    L = len(sequence)
    if L == 0 or seq_len < 3:
        return None

    # ==========================================================
    # 1. Process Saliency & Align Residues
    # ==========================================================
    saliency_a = np.asarray(saliency_a, dtype=np.float32)
    saliency_b = np.asarray(saliency_b, dtype=np.float32)

    # Slice out [CLS] at pos 0 and [EOS] at seq_len - 1
    aa_saliency_a = saliency_a[1 : seq_len - 1]
    aa_saliency_b = saliency_b[1 : seq_len - 1]

    display_len = min(L, len(aa_saliency_a), len(aa_saliency_b))
    if display_len == 0:
        return None

    aa_saliency_a = aa_saliency_a[:display_len]
    aa_saliency_b = aa_saliency_b[:display_len]
    sequence_display = sequence[:display_len]

    x = np.arange(display_len)
    positions_1based = x + 1  # Standard biological 1-based indexing

    # ==========================================================
    # 2. Design System & Palette Config
    # ==========================================================
    # Aesthetic modern palettes
    color_schemes = {
        "teal": {
            "main": "#0F766E",
            "light": "#CCFBF1",
            "highlight": "#D97706",
            "edge": "#115E59",
        },
        "coral": {
            "main": "#BE123C",
            "light": "#FFE4E6",
            "highlight": "#4338CA",
            "edge": "#9F1239",
        },
        "blue": {
            "main": "#2563EB",
            "light": "#DBEAFE",
            "highlight": "#EA580C",
            "edge": "#1D4ED8",
        },
        "purple": {
            "main": "#7C3AED",
            "light": "#EDE9FE",
            "highlight": "#059669",
            "edge": "#6D28D9",
        },
    }

    c_a = (
        color_schemes.get(palette[0], color_schemes["teal"])
        if isinstance(palette, tuple)
        else color_schemes["teal"]
    )
    c_b = (
        color_schemes.get(palette[1], color_schemes["coral"])
        if isinstance(palette, tuple)
        else color_schemes["coral"]
    )

    # Clamp figure width gracefully: readable for short seqs, bounded for long proteins
    fig_width = np.clip(display_len * 0.18 + 4.0, 10.0, 22.0)
    fig, axes = plt.subplots(
        2, 1, figsize=(fig_width, 6.5), sharex=True, sharey=False, dpi=150
    )

    branch_configs = [
        {
            "ax": axes[0],
            "data": aa_saliency_a,
            "title": "Branch A (Trainable)",
            "theme": c_a,
        },
        {
            "ax": axes[1],
            "data": aa_saliency_b,
            "title": "Branch B (Frozen)",
            "theme": c_b,
        },
    ]

    # ==========================================================
    # 3. Draw Subplots
    # ==========================================================
    for cfg in branch_configs:
        ax = cfg["ax"]
        data = cfg["data"]
        theme = cfg["theme"]

        # Base bar plot
        bars = ax.bar(
            x,
            data,
            width=0.85,
            color=theme["main"],
            alpha=0.75,
            edgecolor=theme["edge"],
            linewidth=0.6,
            zorder=3,
        )

        # Baseline & soft grid
        ax.axhline(0, color="#64748B", linewidth=0.8, zorder=2)
        ax.grid(
            axis="y",
            linestyle="--",
            alpha=0.4,
            color="#94A3B8",
            linewidth=0.7,
            zorder=1,
        )

        # Minimalist Spines (Despine top/right)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_color("#64748B")
        ax.spines["bottom"].set_color("#64748B")

        # Y-axis scaling & padding
        y_max = np.max(data) if np.max(data) > 0 else 1.0
        ax.set_ylim(0, y_max * 1.25)
        ax.set_ylabel(
            "Gradient Saliency",
            fontsize=9.5,
            fontweight="bold",
            color="#1E293B",
        )

        # Subplot Title badge
        ax.text(
            0.012,
            0.88,
            cfg["title"],
            transform=ax.transAxes,
            fontsize=10.5,
            fontweight="bold",
            color=theme["main"],
            bbox=dict(
                boxstyle="round,pad=0.35",
                facecolor=theme["light"],
                edgecolor="none",
                alpha=0.9,
            ),
            zorder=4,
        )

        # ======================================================
        # Highlight & Annotate Top-K Residues
        # ======================================================
        top_k_indices = np.argsort(data)[::-1][: min(top_k, len(data))]

        for idx in top_k_indices:
            # Emphasize top bar
            bars[idx].set_color(theme["highlight"])
            bars[idx].set_edgecolor("#9A3412")
            bars[idx].set_alpha(0.95)

            score = data[idx]
            aa = sequence_display[idx]
            pos = positions_1based[idx]

            # Label format: "Met1" or "M1"
            ax.annotate(
                f"{aa}{pos}",
                xy=(idx, score),
                xytext=(0, 5),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=8,
                fontweight="bold",
                color=theme["highlight"],
                zorder=5,
                bbox=dict(
                    boxstyle="round,pad=0.15",
                    facecolor="white",
                    edgecolor=theme["highlight"],
                    alpha=0.85,
                    linewidth=0.5,
                ),
            )

    # ==========================================================
    # 4. Refined X-Axis Formatting
    # ==========================================================
    # Smart tick step: target ~25 readable intervals
    target_ticks = 25
    tick_step = max(1, round(display_len / target_ticks))
    tick_positions = x[::tick_step]

    # Two-tiered tick label: Residue Letter above, 1-based Position below
    tick_labels = [
        f"{sequence_display[i]}\n{positions_1based[i]}" for i in tick_positions
    ]

    axes[1].set_xticks(tick_positions)
    axes[1].set_xticklabels(
        tick_labels, fontsize=7.5, fontweight="medium", color="#334155"
    )
    axes[1].set_xlabel(
        "Residue & Sequence Position (1-indexed)",
        fontsize=10.5,
        fontweight="bold",
        color="#1E293B",
        labelpad=8,
    )

    # Clean tick lines
    for ax in axes:
        ax.tick_params(colors="#64748B", labelsize=8.5, length=4)

    # ==========================================================
    # 5. Overall Title & Layout Finalization
    # ==========================================================
    fig.suptitle(
        "Dual ESM Residue-Level Saliency Comparison",
        fontsize=13,
        fontweight="bold",
        color="#0F172A",
        y=0.98,
    )

    plt.subplots_adjust(
        top=0.91, bottom=0.14, left=0.07, right=0.98, hspace=0.25
    )

    # Safe return (supports both PIL converter or raw figure object)
    if "fig_to_pil" in globals():
        return fig_to_pil(fig)
    return fig

def plot_pooling_attention(attn_weights, sequence, seq_len):
    """
    Multi-head pooling attention heatmap (heads × positions).
    Skips <cls> (pos 0) and <eos> (pos seq_len-1) — amino acids only.
    """
    L = len(sequence)
    if L == 0 or seq_len < 3:
        return None

    # Slice out CLS + EOS: positions 1 .. seq_len-1  along the token axis
    aa_attn = attn_weights[:, 1:seq_len - 1]          # [num_heads, aa_len]
    display_len = min(L, aa_attn.shape[1])
    aa_attn = aa_attn[:, :display_len]
    num_heads = aa_attn.shape[0]

    fig, ax = plt.subplots(figsize=(max(10, display_len * 0.35), 1.2 + num_heads * 0.8))
    im = ax.imshow(aa_attn, aspect='auto', cmap='YlOrRd', interpolation='nearest')

    ax.set_xticks(np.arange(display_len)[::max(1, display_len // 20)])
    ax.set_xticklabels([sequence[i] for i in range(0, display_len, max(1, display_len // 20))], fontsize=7)
    ax.set_yticks(np.arange(num_heads))
    ax.set_yticklabels([f'Head {i+1}' for i in range(num_heads)], fontsize=10)
    ax.set_xlabel('Sequence Position', fontsize=11)
    ax.set_title('Multi-Head Pooling Attention (amino acids only)', fontsize=13, fontweight='bold')
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label('Attention Weight', fontsize=9)
    fig.tight_layout()
    return fig_to_pil(fig)


def plot_cross_attention(cross_attn_map, sequence, seq_len):
    """
    Cross-attention maps: one subplot per head.
    Skips <cls> (pos 0) and <eos> (pos seq_len-1) on both axes.
    """
    L = len(sequence)
    if L == 0 or seq_len < 3:
        return None

    # Slice out CLS + EOS on both query and key axes
    aa_map = cross_attn_map[:, 1:seq_len - 1, 1:seq_len - 1]   # [H, aa_len, aa_len]
    display_len = min(L, aa_map.shape[1])
    aa_map = aa_map[:, :display_len, :display_len]
    num_heads = aa_map.shape[0]

    ncols = min(4, num_heads)
    nrows = int(np.ceil(num_heads / ncols))
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(3.5 * ncols, 3.2 * nrows),
                             squeeze=False)

    tick_step = max(1, display_len // 15)
    tick_positions = np.arange(0, display_len, tick_step)
    tick_labels = [sequence[i] for i in tick_positions]

    for h in range(num_heads):
        r, c = divmod(h, ncols)
        ax = axes[r][c]
        im = ax.imshow(aa_map[h], aspect='auto', cmap='Blues',
                       interpolation='nearest')
        ax.set_title(f'Head {h+1}', fontsize=10, fontweight='bold')
        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, fontsize=6, rotation=45)
        ax.set_yticks(tick_positions)
        ax.set_yticklabels(tick_labels, fontsize=6)
        if c == 0:
            ax.set_ylabel('Key (frozen ESM)', fontsize=9)
        if r == nrows - 1:
            ax.set_xlabel('Query (trainable ESM)', fontsize=9)

    # Hide unused subplots
    for h in range(num_heads, nrows * ncols):
        r, c = divmod(h, ncols)
        axes[r][c].set_visible(False)

    fig.suptitle('Cross-Attention: Trainable → Frozen ESM (amino acids only)', fontsize=13, fontweight='bold', y=1.01)
    fig.tight_layout()
    return fig_to_pil(fig)


VALID_AA = set('ACDEFGHIKLMNPQRSTVWY')

def analyze_peptide(sequence: str, target_class_name: str = "auto"):
    """
    Full analysis pipeline: prediction + all 4 explainability views.

    Parameters
    ----------
    sequence : str
        Raw amino-acid sequence.
    target_class_name : str
        Functional class to explain with saliency.
        "auto" → use the highest-probability class.
        Otherwise a name from endpoints_functional (e.g. "anti-cancer").
    """
    # --- Validation ---
    if not sequence or not sequence.strip():
        return (
            "⚠️  Please enter a peptide sequence.",
            None, None, None, None,
        )

    sequence = sequence.strip().upper()
    if not all(aa in VALID_AA for aa in sequence):
        invalid = [aa for aa in sequence if aa not in VALID_AA]
        return (
            f"⚠️  Invalid amino acid(s): {', '.join(set(invalid))}",
            None, None, None, None,
        )

    # --- Tokenize ---
    encodings = esm_tokenizer(
        [sequence],
        add_special_tokens=True,
        max_length=100,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt',
    )
    input_ids = encodings['input_ids'].to(DEVICE)
    attention_mask = encodings['attention_mask'].to(DEVICE)
    seq_len = int(attention_mask.sum().item())

    # --- 1. Prediction ---
    results = batched_sequence_classifier([sequence], model)
    probs = results[0]['gated_probs']

    pred_lines = []
    for cls, prob in probs.items():
        th = thresholds.get(cls, 0.5)
        bar = '█' * int(prob * 20)
        status = '✅ ACTIVE ' if prob >= th else '⬜ inactive'
        pred_lines.append(f"{cls:28s} {bar:20s} {prob:.4f}  {status}")
    pred_text = (
        f"📋  Sequence: {sequence}  (length={len(sequence)})\n"
        f"{'─'*65}\n" + "\n".join(pred_lines) + f"\n{'─'*65}\n"
        f"🔑  Threshold source: best validation epoch (MCC-optimized)\n"
    )

    # --- 2. Saliency Map (class-wise) ---
    if target_class_name == "auto" or target_class_name not in endpoints_functional:
        target_idx = max(
            range(len(endpoints_functional)),
            key=lambda i: probs[endpoints_functional[i]],
        )
        target_cls = endpoints_functional[target_idx]
        pred_text += f"🎯  Saliency target: {target_cls}  (auto – highest prob)\n"
    else:
        target_idx = endpoints_functional.index(target_class_name)
        target_cls = target_class_name
        pred_text += f"🎯  Saliency target: {target_cls}  (manual selection)\n"

    try:

        saliency_a, saliency_b = residue_saliency_dual(
            model,
            input_ids,
            attention_mask,
            target_class=target_idx
        )

        saliency_img = plot_saliency(
            sequence,
            saliency_a,
            saliency_b,
            seq_len
        )

    except Exception as e:

        saliency_img = None

        pred_text += (
            f"\n⚠️ Saliency failed: {e}"
        )

    # --- 3. Pooling Attention ---
    try:
        attn_w = get_pooling_attention(model, input_ids, attention_mask)
        attn_img = plot_pooling_attention(attn_w, sequence, seq_len)
    except Exception as e:
        attn_img = None
        pred_text += f"\n⚠️  Pooling attention failed: {e}"

    # --- 4. Cross-Attention ---
    try:
        cross_w = get_cross_attention_map(model, input_ids, attention_mask)
        cross_img = plot_cross_attention(cross_w, sequence, seq_len)
    except Exception as e:
        cross_img = None
        pred_text += f"\n⚠️  Cross-attention failed: {e}"

    # --- 5. CNN Motifs ---
    try:
        cnn_img = plot_cnn_activations(model, input_ids, sequence, seq_len)
    except Exception as e:
        cnn_img = None
        pred_text += f"\n⚠️  CNN motifs failed: {e}"

    return pred_text, saliency_img, attn_img, cross_img, cnn_img


def analyze_batch(sequences_text: str, uploaded_file, tta_passes: float = 1):
    """
    Batch endpoint: submit MANY sequences at once.

    - Input: sequences pasted in the textbox, OR an uploaded file
      (.txt / .csv). If a file is uploaded, it takes priority.
    - Internally calls batched_sequence_classifier with a hardcoded
      batch_size=256 (never displayed on the website).
    - tta_passes is clamped to [1, 5] inclusive.
    - Only the first 100 results are rendered on the page (to keep the
      website responsive); the downloadable CSV always contains ALL results.

    Returns:
        summary text, display table (≤100 rows), CSV file path (all rows)
    """
    import tempfile

    class_cols = list(thresholds.keys())   # functional endpoints + 'non-functional'
    MAX_DISPLAY = 100                      # max rows rendered on the page

    # --- Clamp TTA passes to 1..5 ---
    tta = int(round(float(tta_passes)))
    tta = max(1, min(5, tta))

    # --- Input source: uploaded file wins over the textbox ---
    if uploaded_file is not None:
        path = uploaded_file if isinstance(uploaded_file, str) else getattr(uploaded_file, 'name', None)
        if not path:
            return "⚠️  Could not read uploaded file.", pd.DataFrame(), None
        try:
            if str(path).lower().endswith('.csv'):
                df_in = pd.read_csv(path)
                col = df_in.columns[0]
                sequences_text = "\n".join(df_in[col].astype(str).tolist())
            else:
                with open(path, 'r', encoding='utf-8', errors='ignore') as fh:
                    sequences_text = fh.read()
        except Exception as e:
            return f"⚠️  Could not read uploaded file: {e}", pd.DataFrame(), None

    # --- Parse & deduplicate (order preserved) ---
    if not sequences_text or not str(sequences_text).strip():
        return "⚠️  Please paste at least one sequence or upload a file.", pd.DataFrame(), None

    tokens = str(sequences_text).replace(',', ' ').split()
    sequences, seen = [], set()
    for tok in tokens:
        s = tok.strip().upper()
        if s and s not in seen:
            seen.add(s)
            sequences.append(s)

    if not sequences:
        return "⚠️  No sequences found in the input.", pd.DataFrame(), None

    MAX_INPUT = 10000
    trunc_note = ""
    if len(sequences) > MAX_INPUT:
        sequences = sequences[:MAX_INPUT]
        trunc_note = f"⚠️  Input truncated to the first {MAX_INPUT} sequences.\n\n"

    # --- Validation ---
    bad_lines = []
    for i, s in enumerate(sequences):
        bad = sorted(set(aa for aa in s if aa not in VALID_AA))
        if bad:
            short = s[:25] + ('…' if len(s) > 25 else '')
            bad_lines.append(f"  #{i+1}: {short}  → invalid: {', '.join(bad)}")
    if bad_lines:
        msg = "⚠️  Invalid amino acid(s):\n" + "\n".join(bad_lines[:20])
        if len(bad_lines) > 20:
            msg += f"\n  …and {len(bad_lines) - 20} more invalid sequences."
        return msg, pd.DataFrame(), None

    # --- Inference (batch_size is hardcoded to 256 internally) ---
    results = batched_sequence_classifier(
        sequences,
        model,
        batch_size=256,
        tta_passes=tta,
    )

    # --- Build summary text (first MAX_DISPLAY sequences only) ---
    lines = [
        f"📦  {len(sequences)} sequence(s) analysed",
        f"🔄  TTA passes = {tta}",
        "─" * 70,
    ]
    rows = []
    for n, (s, r) in enumerate(zip(sequences, results), start=1):
        probs = r['gated_probs']
        active = [
            (c, p) for c, p in probs.items()
            if c != 'non-functional' and p >= thresholds.get(c, 0.5)
        ]
        active.sort(key=lambda t: -t[1])

        if n <= MAX_DISPLAY:
            if active:
                act_str = ', '.join(f"{c}={p:.3f}" for c, p in active)
            else:
                act_str = f"non-functional={probs['non-functional']:.3f}"
            disp = s if len(s) <= 32 else s[:32] + '…'
            lines.append(f"{n:>5}. {disp}")
            lines.append(f"       ✅ {act_str}")

        # Table cells: append " (active)" to classes above their threshold
        row = [s]
        for c in class_cols:
            p = probs[c]
            th = thresholds.get(c, 0.5)
            row.append(f"{p:.4f} (active)" if p >= th else f"{p:.4f}")
        row.append(', '.join(name for name, _ in active) if active else '')
        rows.append(row)

    if len(sequences) > MAX_DISPLAY:
        lines += [
            "─" * 70,
            f"⚠️  Showing the first {MAX_DISPLAY} of {len(sequences)} sequences on the page "
            "— download the CSV for the complete table.",
        ]

    text_out = trunc_note + "\n".join(lines)
    df_full = pd.DataFrame(rows, columns=['sequence'] + class_cols + ['active_classes'])
    df_display = df_full.head(MAX_DISPLAY)

    # --- Write CSV with ALL results for download ---
    tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.csv', prefix='batch_results_')
    csv_path = tmp.name
    tmp.close()
    df_full.to_csv(csv_path, index=False)

    return text_out, df_display, csv_path


# ---------- Gradio Blocks Interface ----------

with gr.Blocks(
    title="Peptide Explainability Dashboard",
    theme=gr.themes.Soft(primary_hue="emerald"),
    css="""
    .pred-box textarea { font-family: 'Courier New', monospace !important; font-size: 13px !important; }
    """
) as demo:

    # --- Header ---
    gr.Markdown("""
    # 🔬  Peptide Functional Classifier — Explainability Dashboard

    Enter a peptide sequence below to predict its functional classes and explore
    **what the model is looking at** via four interpretability lenses.
    Use the **📦 Batch Predictions** tab to submit many sequences at once.
    """)

    # --- Input Row ---
    with gr.Row(equal_height=True):
        seq_input = gr.Textbox(
            label="🧪  Peptide Sequence",
            placeholder="e.g.  GLFDVIKKIAESI  or  FLPVLAGLTPSIVPKLVCLLTKKC",
            lines=2,
            scale=5,
            elem_id="seq-input",
        )
        with gr.Column(scale=1, min_width=100):
            submit_btn = gr.Button("🔍  Analyze", variant="primary", size="lg")
            gr.Markdown("")  # spacer

    # --- Example peptides ---
    gr.Examples(
        examples=[
            "GLFDVIKKIAESI",
            "FLPVLAGLTPSIVPKLVCLLTKKC",
            "KWKLFKKIGAVLKVL",
            "GIGAVLKVLTTGLPALISWIKRKRQQ",
        ],
        inputs=seq_input,
        label="💡  Try an example",
    )

    # --- Results Tabs ---
    with gr.Tabs():
        with gr.TabItem("📊  Predictions"):
            pred_output = gr.Textbox(
                label="Classification Results",
                lines=18,
                max_lines=25,
                elem_classes="pred-box",
            )

        with gr.TabItem("🎯  Residue Saliency"):
            saliency_class = gr.Dropdown(
                choices=["auto"] + endpoints_functional,
                value="auto",
                label="Target class for saliency",
                info="'auto' = highest-probability class; or pick a specific class",
            )
            saliency_output = gr.Image(
                label="Which amino acids drive the prediction? (gradient w.r.t. ESM hidden states)",
                type="pil",
            )

        with gr.TabItem("🧠  Pooling Attention"):
            attn_output = gr.Image(
                label="How do attention heads distribute focus across the sequence?",
                type="pil",
            )

        with gr.TabItem("🔗  Cross-Attention"):
            cross_attn_output = gr.Image(
                label="Where does the trainable ESM look in the frozen ESM?",
                type="pil",
            )

        with gr.TabItem("🧬  CNN Motifs"):
            motifs_output = gr.Image(
                label="Which k-mer patterns activate the convolutional filters?",
                type="pil",
            )

        with gr.TabItem("📦  Batch Predictions"):
            batch_input = gr.Textbox(
                label="🧪  Sequences — one per line (or comma/space separated)",
                placeholder="GLFDVIKKIAESI\nFLPVLAGLTPSIVPKLVCLLTKKC\nKWKLFKKIGAVLKVL\nGIGAVLKVLTTGLPALISWIKRKRQQ",
                lines=8,
                max_lines=20,
                elem_classes="pred-box",
            )
            batch_file = gr.File(
                label="📁  …or upload a file (.txt / .csv — one sequence per line)",
                file_count="single",
                file_types=[".txt", ".csv"],
                type="filepath",
            )
            with gr.Row(equal_height=True):
                tta_slider = gr.Slider(
                    minimum=1,
                    maximum=5,
                    step=1,
                    value=1,
                    label="🔄  TTA passes (1–5)",
                    info="1 = fast · 5 = most stable probabilities",
                    scale=3,
                )
                batch_btn = gr.Button("🚀  Run Batch", variant="primary", size="lg", scale=1)
            batch_output = gr.Textbox(
                label="Batch Results",
                lines=14,
                max_lines=30,
                elem_classes="pred-box",
            )
            batch_table = gr.Dataframe(
                label="📋  Probability table (max 100 rows rendered; active classes marked)",
                interactive=False,
            )
            batch_download = gr.File(
                label="⬇️  Download full results (CSV — all sequences)",
            )

    # --- Example batch input ---
    gr.Examples(
        examples=[
            "GLFDVIKKIAESI\nFLPVLAGLTPSIVPKLVCLLTKKC\nKWKLFKKIGAVLKVL",
            "GIGAVLKVLTTGLPALISWIKRKRQQ\nGLFDVIKKIAESI\nKWKLFKKIGAVLKVL",
        ],
        inputs=batch_input,
        label="💡  Try a batch",
    )

    # --- Wire up: single sequence ---
    submit_btn.click(
        fn=analyze_peptide,
        inputs=[seq_input, saliency_class],
        outputs=[pred_output, saliency_output, attn_output, cross_attn_output, motifs_output],
    )

    # Allow Enter key
    seq_input.submit(
        fn=analyze_peptide,
        inputs=[seq_input, saliency_class],
        outputs=[pred_output, saliency_output, attn_output, cross_attn_output, motifs_output],
    )

    # Auto-update saliency when class dropdown changes (no need to re-click Analyze)
    saliency_class.change(
        fn=analyze_peptide,
        inputs=[seq_input, saliency_class],
        outputs=[pred_output, saliency_output, attn_output, cross_attn_output, motifs_output],
    )

    # --- Wire up: batch prediction ---
    batch_btn.click(
        fn=analyze_batch,
        inputs=[batch_input, batch_file, tta_slider],
        outputs=[batch_output, batch_table, batch_download],
    )

    # Allow Enter key in the batch box too
    batch_input.submit(
        fn=analyze_batch,
        inputs=[batch_input, batch_file, tta_slider],
        outputs=[batch_output, batch_table, batch_download],
    )

    # --- Footer ---
    gr.Markdown("""
    ---
    ### 📖  Interpretation Guide

    | View | What it shows | How to read it |
    |------|--------------|----------------|
    | **Saliency** | Gradient of the selected class w.r.t. ESM hidden states | Taller bars = residues most influential for that class. Use the dropdown to compare classes! |
    | **Pooling Attention** | Learned attention weights before the final classification head | Each head may specialize in different regions |
    | **Cross-Attention** | Interaction between the trainable (query) and frozen (key) ESM encoders | Bright spots = positions the model links together |
    | **CNN Motifs** | Summed filter activations across 3 kernel sizes (3, 5, 7) | Peaks = sequence motifs that trigger the convolutional filters |
    | **Batch Predictions** | Run classification over many sequences in one call (TTA 1–5) | Paste sequences or upload a file; the page shows max 100 rows, the CSV has all results. Cells above threshold are marked **(active)** |
    """)

print("✅ Gradio dashboard defined.  Run the next cell to launch.")

/tmp/ipykernel_1029277/787032458.py:672: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


✅ Gradio dashboard defined.  Run the next cell to launch.


In [44]:
# =====================================================================
# 6. Launch the Dashboard
# =====================================================================
# Use share=True for a public link (good for demos / colab-style sharing)
# Use share=False for local-only access

demo.launch(
    server_name="0.0.0.0",   # accessible from other machines on the network
    server_port=7860,
    share=False,              # set True for a public gradio.live link
    show_error=True,
    root_path="/pepmtl"
)


OSError: Cannot find empty port in range: 7860-7860. You can specify a different port by setting the GRADIO_SERVER_PORT environment variable or passing the `server_port` parameter to `launch()`.

In [45]:

def fig_to_pil(fig, dpi=130):
    """Convert a matplotlib Figure to an EPS BytesIO object for Gradio."""
    buf = BytesIO()
    fig.savefig(buf, format='eps', dpi=600, bbox_inches='tight',
                pad_inches=0.25,
                facecolor='white', edgecolor='none')

    buf.seek(0)
    plt.close(fig)
    return buf

In [46]:
result = analyze_peptide("ILPWKWPWWPWRR")

with open("indolicidin_esm.eps", "wb") as f:
    f.write(result[1].getvalue())

# 2. Save the buffer content to an EPS file
with open("indolicidin_cnn.eps", "wb") as f:
    f.write(result[4].getvalue())

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


In [47]:
result = analyze_peptide("HAEGTFTSDVSSYLEGQAAKEFIAWLVKGR")

with open("glp_esm.eps", "wb") as f:
    f.write(result[1].getvalue())

# 2. Save the buffer content to an EPS file
with open("glp_cnn.eps", "wb") as f:
    f.write(result[4].getvalue())

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.
